<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V14_MultiCityCleaning_WeatherMapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ══════════════════════════════════════════════════════════════
# 0 — Google Drive & Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"
!unzip -q "/content/data.zip" -d "/content"
!rm "/content/data.zip"
!rm -rf "/content/_MACOSX"

Mounted at /content/drive


In [7]:
# ══════════════════════════════════════════════════════════════
# Batch-Cleaning mehrerer Städte
#
# Maßnahmen:
#   [1] geo_information: WKB-Hex → lat/lon dekodieren, 'location' droppen
#   [2] demand/<CITY>: nur station_type='real' behalten
#   [3] demand/<CITY>: Stationen mit max. Datenlücke > 90 Tage entfernen
#   [4] demand/<CITY>: fehlende Tage (Demand-Lücken auf Stadtebene) entfernen
#
# Output:
#   /content/drive/MyDrive/BA_Colab/cleaned/demand/<CITY>/demand_cleaned.parquet
#   /content/drive/MyDrive/BA_Colab/cleaned/geo_information/geo_information.parquet
# ══════════════════════════════════════════════════════════════

import os, glob, re
import pandas as pd
import pyarrow.parquet as pq
from shapely import wkb as shapely_wkb

# ── Pfade ─────────────────────────────────────────────────────
DATA_BASE    = '/content/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

# ── Gewünschte Städte ────────────────────────────────────────
CITIES = [
    'Gießen',
    'Leverkusen',
    'Braunschweig',
    'Dortmund',
    'Duisburg',
    'Bochum',
    'Essen',
    'Offenburg'
]

# ── Station-Typ Klassifikation (identisch zur Pipeline) ──────
def classify_station(name):
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):
        return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):
        return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):
        return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):
        return 'only_nums'
    return 'real'

# ── Station Names laden ──────────────────────────────────────
station_names_raw = pd.read_parquet(f'{DATA_BASE}/station_names/station_names.parquet')
station_names_raw = station_names_raw.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names_raw['station_type'] = station_names_raw['station_name'].apply(classify_station)
type_lookup = station_names_raw.set_index('station_name_id')['station_type'].to_dict()

print('╔════════════════════════════════════════════════════════════╗')
print('║   BATCH CLEANING — mehrere Städte                         ║')
print('╚════════════════════════════════════════════════════════════╝')

# ────────────────────────────────────────────────
# [1] GEO_INFORMATION: WKB → lat/lon
# ────────────────────────────────────────────────
print('\n[1] geo_information: WKB → lat/lon ...')

geo_out_dir = f'{CLEANED_BASE}/geo_information'
os.makedirs(geo_out_dir, exist_ok=True)

geo_raw = pq.read_table(f'{DATA_BASE}/geo_information').to_pandas()

if 'location' in geo_raw.columns:
    geom = geo_raw['location'].apply(lambda x: shapely_wkb.loads(bytes.fromhex(x)))
    geo_raw['latitude']  = geom.apply(lambda g: round(g.y, 6))
    geo_raw['longitude'] = geom.apply(lambda g: round(g.x, 6))
    geo_clean = geo_raw.drop(columns=['location'])
else:
    geo_clean = geo_raw.copy()
    print('  Hinweis: Spalte "location" nicht vorhanden – geo_information bereits dekodiert?')

geo_clean.to_parquet(f'{geo_out_dir}/geo_information.parquet', index=False)
print(f'  ✅ Gespeichert: {len(geo_clean):,} Einträge')
print(f'  Pfad: {geo_out_dir}/geo_information.parquet')

# ────────────────────────────────────────────────
# [2+3+4] DEMAND / mehrere Städte
# ────────────────────────────────────────────────
summary = []

for CITY in CITIES:
    print('\n' + '─' * 68)
    print(f'CITY: {CITY}')
    print('─' * 68)

    files = glob.glob(f'{DATA_BASE}/demand/{CITY}/**/*.parquet', recursive=True)
    if not files:
        files = glob.glob(f'{DATA_BASE}/demand/{CITY}/*.parquet')

    if not files:
        print(f'  ❌ Keine Parquet-Dateien gefunden für: {CITY}')
        summary.append({
            'city': CITY,
            'status': 'missing',
            'raw_rows': 0,
            'final_rows': 0,
            'raw_stations': 0,
            'final_stations': 0,
            'removed_gap_stations': 0,
            'removed_missing_days': 0
        })
        continue

    print(f'  Parquet-Dateien gefunden: {len(files)}')

    cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
            'location_id', 'n_lends', 'n_returns']

    demand = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
    demand['timestamp'] = pd.to_datetime(demand['timestamp'], utc=True)
    demand['station_type'] = demand['station_name_id'].map(type_lookup).fillna('unknown')
    demand['total_demand'] = demand['n_lends'] + demand['n_returns']

    raw_rows = len(demand)
    raw_stations = demand['station_id'].nunique()

    print(f'  Roh: {raw_rows:,} Zeilen, {raw_stations} Stationen')

    # [2] Nur real Stationen
    before = len(demand)
    demand = demand[demand['station_type'] == 'real'].copy()
    print(f'  [2] Nur real stations: {len(demand):,} Zeilen '
          f'(entfernt: {before - len(demand):,}) | '
          f'{demand["station_id"].nunique()} Stationen')

    # [3] Stationen mit max. Datenlücke > 90 Tage entfernen
    print('  [3] Berechne Datenlücken pro Station ...')
    demand['date'] = demand['timestamp'].dt.date

    def max_gap(dates):
        s = sorted(set(pd.to_datetime(d) for d in dates))
        if len(s) < 2:
            return 0
        return max((s[i+1] - s[i]).days - 1 for i in range(len(s)-1))

    station_gaps = (
        demand.groupby('station_id')['date']
        .apply(max_gap)
        .reset_index()
        .rename(columns={'date': 'max_gap_days'})
    )

    bad_stations = station_gaps[station_gaps['max_gap_days'] > 90]['station_id'].tolist()
    demand = demand[~demand['station_id'].isin(bad_stations)].copy()

    print(f'  Stationen mit Lücke > 90d entfernt: {len(bad_stations)}')
    if len(bad_stations) > 0:
        print(f'    IDs: {bad_stations}')

    print(f'  Nach Lücken-Filter: {demand["station_id"].nunique()} Stationen, {len(demand):,} Zeilen')

    # [4] Fehlende Tage auf Stadtebene entfernen
    print('  [4] Fehlende Tage auf Stadtebene entfernen ...')
    daily_city = demand.groupby('date')['total_demand'].sum().reset_index()
    all_dates = pd.date_range(
        demand['timestamp'].min().date(),
        demand['timestamp'].max().date(),
        freq='D'
    )
    existing_dates = set(pd.Timestamp(d) for d in daily_city['date'])
    missing_dates = [d for d in all_dates if d not in existing_dates]

    if missing_dates:
        missing_date_set = {d.date() for d in missing_dates}
        demand = demand[~demand['date'].isin(missing_date_set)].copy()
        print(f'  Entfernte Tage ({len(missing_dates)}): {[str(d.date()) for d in missing_dates]}')
    else:
        print('  Keine fehlenden Tage gefunden.')

    # Hilfsspalten droppen
    demand = demand.drop(columns=['station_type', 'total_demand', 'date'])

    # Speichern
    demand_out_dir = f'{CLEANED_BASE}/demand/{CITY}'
    os.makedirs(demand_out_dir, exist_ok=True)

    out_path = f'{demand_out_dir}/demand_cleaned.parquet'
    demand.to_parquet(out_path, index=False)

    final_rows = len(demand)
    final_stations = demand['station_id'].nunique()

    print(f'  ✅ Gespeichert: {final_rows:,} Zeilen, {final_stations} Stationen')
    print(f'  Pfad: {out_path}')

    summary.append({
        'city': CITY,
        'status': 'ok',
        'raw_rows': raw_rows,
        'final_rows': final_rows,
        'raw_stations': raw_stations,
        'final_stations': final_stations,
        'removed_gap_stations': len(bad_stations),
        'removed_missing_days': len(missing_dates)
    })

# ────────────────────────────────────────────────
# Zusammenfassung
# ────────────────────────────────────────────────
print('\n╔════════════════════════════════════════════════════════════╗')
print('║   CLEANING ABGESCHLOSSEN                                  ║')
print('╚════════════════════════════════════════════════════════════╝')

summary_df = pd.DataFrame(summary)
display(summary_df)

print('\nOutput-Basis:')
print(f'  {CLEANED_BASE}/')
print('  ├── geo_information/geo_information.parquet')
print('  └── demand/<CITY>/demand_cleaned.parquet')

╔════════════════════════════════════════════════════════════╗
║   BATCH CLEANING — mehrere Städte                         ║
╚════════════════════════════════════════════════════════════╝

[1] geo_information: WKB → lat/lon ...
  ✅ Gespeichert: 300,508 Einträge
  Pfad: /content/drive/MyDrive/BA_Colab/cleaned/geo_information/geo_information.parquet

────────────────────────────────────────────────────────────────────
CITY: Gießen
────────────────────────────────────────────────────────────────────
  Parquet-Dateien gefunden: 36
  Roh: 1,082,120 Zeilen, 37560 Stationen
  [2] Nur real stations: 1,041,980 Zeilen (entfernt: 40,140) | 67 Stationen
  [3] Berechne Datenlücken pro Station ...
  Stationen mit Lücke > 90d entfernt: 2
    IDs: ['381684271', '6510981']
  Nach Lücken-Filter: 65 Stationen, 1,024,199 Zeilen
  [4] Fehlende Tage auf Stadtebene entfernen ...
  Entfernte Tage (32): ['2023-06-29', '2023-06-30', '2023-07-01', '2023-07-02', '2023-07-03', '2023-07-04', '2023-07-05', '2023-07-

,city,status,raw_rows,final_rows,raw_stations,final_stations,removed_gap_stations,removed_missing_days
0,Gießen,ok,1082120,1024199,37560,65,2,32
1,Leverkusen,ok,631820,600845,26535,149,3,32
2,Braunschweig,ok,420448,397269,19034,159,1,0
3,Dortmund,ok,345749,334022,10906,100,0,0
4,Duisburg,ok,253832,246265,6952,58,0,0
5,Bochum,ok,203035,195168,7264,95,0,0
6,Essen,ok,156162,145190,7729,92,2,0
7,Offenburg,ok,151580,143369,7340,28,0,12



Output-Basis:
  /content/drive/MyDrive/BA_Colab/cleaned/
  ├── geo_information/geo_information.parquet
  └── demand/<CITY>/demand_cleaned.parquet


In [8]:
import os, glob, re
import pandas as pd

DATA_BASE    = '/content/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

CITIES = [
    'Gießen',
    'Leverkusen',
    'Braunschweig',
    'Dortmund',
    'Duisburg',
    'Bochum',
    'Essen',
    'Offenburg'
]

def classify_station(name):
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):
        return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):
        return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):
        return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):
        return 'only_nums'
    return 'real'

station_names_raw = pd.read_parquet(f'{DATA_BASE}/station_names/station_names.parquet')
station_names_raw = station_names_raw.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names_raw['station_type'] = station_names_raw['station_name'].apply(classify_station)
type_lookup = station_names_raw.set_index('station_name_id')['station_type'].to_dict()

def max_gap(dates):
    s = sorted(set(pd.to_datetime(d) for d in dates))
    if len(s) < 2:
        return 0
    return max((s[i+1] - s[i]).days - 1 for i in range(len(s)-1))

results = []

for city in CITIES:
    path = f'{CLEANED_BASE}/demand/{city}/demand_cleaned.parquet'

    if not os.path.exists(path):
        results.append({
            'city': city,
            'exists': False,
            'rows': 0,
            'stations': 0,
            'only_real_ok': False,
            'max_station_gap_ok': False,
            'missing_days_ok': False,
            'status': 'FILE MISSING'
        })
        continue

    df = pd.read_parquet(path)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['date'] = df['timestamp'].dt.date
    df['station_type'] = df['station_name_id'].map(type_lookup).fillna('unknown')
    df['total_demand'] = df['n_lends'] + df['n_returns']

    # Check 1: nur real stations?
    only_real_ok = (df['station_type'] == 'real').all()

    # Check 2: max station gap <= 90?
    station_gaps = df.groupby('station_id')['date'].apply(max_gap)
    max_station_gap = int(station_gaps.max()) if len(station_gaps) else None
    max_station_gap_ok = (max_station_gap is not None and max_station_gap <= 90)

    # Check 3: keine fehlenden Tage auf Stadtebene?
    daily_city = df.groupby('date')['total_demand'].sum().reset_index()
    all_dates = pd.date_range(df['timestamp'].min().date(), df['timestamp'].max().date(), freq='D')
    existing_dates = set(pd.Timestamp(d) for d in daily_city['date'])
    missing_dates = [d for d in all_dates if d not in existing_dates]
    missing_days_ok = (len(missing_dates) == 0)

    status = 'OK' if (only_real_ok and max_station_gap_ok and missing_days_ok) else 'CHECK'

    results.append({
        'city': city,
        'exists': True,
        'rows': len(df),
        'stations': df['station_id'].nunique(),
        'only_real_ok': only_real_ok,
        'max_station_gap_days': max_station_gap,
        'max_station_gap_ok': max_station_gap_ok,
        'missing_days_n': len(missing_dates),
        'missing_days_ok': missing_days_ok,
        'status': status
    })

results_df = pd.DataFrame(results)
display(results_df)

KeyboardInterrupt: 

In [9]:
# ══════════════════════════════════════════════════════════════
# Mapping: ALLE Städte aus /content/data/demand/ → Wetterstation
#
# Auswahlregel:
#   Für jede Stadt wird die Wetterstation gewählt, deren mittlere
#   Distanz zu allen Bike-Standorten der gecleanten Stadt minimal ist.
#
# Output-Spalten:
#   city
#   nearest_weather_location_id
#   weather_station_city_name
#   weather_city_station_FD
#   w_lat
#   w_long
#   mean_distance_km
#
# Speichert nach:
#   /content/drive/MyDrive/BA_Colab/cleaned/city_to_weather_station_mapping.parquet
#   /content/drive/MyDrive/BA_Colab/cleaned/city_to_weather_station_mapping.csv
# ══════════════════════════════════════════════════════════════

import os
import glob
import numpy as np
import pandas as pd

DATA_BASE    = '/content/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

# ── Haversine Distanz in km ───────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

print('╔════════════════════════════════════════════════════════════╗')
print('║   MAPPING ALLER STÄDTE → REPRÄSENTATIVSTE WETTERSTATION   ║')
print('╚════════════════════════════════════════════════════════════╝')

# ── Alle Städte aus data/demand ziehen ───────────────────────
all_city_dirs = sorted([
    d for d in glob.glob(f'{DATA_BASE}/demand/*')
    if os.path.isdir(d)
])
CITIES = [os.path.basename(d) for d in all_city_dirs]

print(f'\nGefundene Städte in {DATA_BASE}/demand/: {len(CITIES)}')
print(CITIES[:10], '...' if len(CITIES) > 10 else '')

# ── Geo laden ────────────────────────────────────────────────
geo_path_cleaned = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
geo_path_raw     = f'{DATA_BASE}/geo_information'

if os.path.exists(geo_path_cleaned):
    geo = pd.read_parquet(geo_path_cleaned)
    print(f'\nGeo geladen aus cleaned: {geo_path_cleaned}')
else:
    geo = pd.read_parquet(geo_path_raw)
    print(f'\nGeo geladen aus raw: {geo_path_raw}')

geo_cols_needed = ['location_id', 'latitude', 'longitude', 'city_name', 'federal_state_name']
geo = geo[[c for c in geo_cols_needed if c in geo.columns]].drop_duplicates(subset=['location_id']).copy()
geo = geo.dropna(subset=['latitude', 'longitude']).copy()

# ── Wetter-Standorte bestimmen ───────────────────────────────
weather_path = f'{DATA_BASE}/weather/weather.parquet'
weather = pd.read_parquet(weather_path, columns=['location_id'])
weather_locations = weather[['location_id']].drop_duplicates().copy()

weather_geo = weather_locations.merge(
    geo,
    on='location_id',
    how='left'
).rename(columns={
    'location_id': 'weather_location_id',
    'latitude': 'weather_latitude',
    'longitude': 'weather_longitude',
    'city_name': 'weather_city_name',
    'federal_state_name': 'weather_federal_state_name'
})

weather_geo = weather_geo.dropna(subset=['weather_latitude', 'weather_longitude']).reset_index(drop=True)

print(f'Wetterstationen mit Geo-Koordinaten: {len(weather_geo):,}')

results = []

for city in CITIES:
    cleaned_path = f'{CLEANED_BASE}/demand/{city}/demand_cleaned.parquet'

    if not os.path.exists(cleaned_path):
        print(f'⚠️ Übersprungen (kein cleaned demand): {city}')
        continue

    try:
        demand = pd.read_parquet(cleaned_path, columns=['location_id'])
    except Exception as e:
        print(f'⚠️ Fehler beim Laden von {city}: {e}')
        continue

    bike_locations = demand[['location_id']].drop_duplicates().copy()

    bike_geo = bike_locations.merge(
        geo,
        on='location_id',
        how='left'
    ).rename(columns={
        'location_id': 'bike_location_id',
        'latitude': 'bike_latitude',
        'longitude': 'bike_longitude'
    })

    bike_geo = bike_geo.dropna(subset=['bike_latitude', 'bike_longitude']).reset_index(drop=True)

    if bike_geo.empty or weather_geo.empty:
        print(f'⚠️ Übersprungen (fehlende Geo-Daten): {city}')
        continue

    bike_lats = bike_geo['bike_latitude'].to_numpy()
    bike_lons = bike_geo['bike_longitude'].to_numpy()

    station_scores = []

    for _, wrow in weather_geo.iterrows():
        distances = haversine_km(
            bike_lats,
            bike_lons,
            wrow['weather_latitude'],
            wrow['weather_longitude']
        )

        station_scores.append({
            'nearest_weather_location_id': int(wrow['weather_location_id']),
            'weather_station_city_name': wrow.get('weather_city_name', None),
            'weather_city_station_FD': wrow.get('weather_federal_state_name', None),
            'w_lat': float(wrow['weather_latitude']),
            'w_long': float(wrow['weather_longitude']),
            'mean_distance_km': float(np.mean(distances))
        })

    score_df = pd.DataFrame(station_scores).sort_values(
        by='mean_distance_km',
        ascending=True
    ).reset_index(drop=True)

    best = score_df.iloc[0]

    results.append({
        'city': city,
        'nearest_weather_location_id': int(best['nearest_weather_location_id']),
        'weather_station_city_name': best['weather_station_city_name'],
        'weather_city_station_FD': best['weather_city_station_FD'],
        'w_lat': round(float(best['w_lat']), 6),
        'w_long': round(float(best['w_long']), 6),
        'mean_distance_km': round(float(best['mean_distance_km']), 3)
    })

    print(
        f"✅ {city}: Wetterstation {int(best['nearest_weather_location_id'])} "
        f"({best['weather_station_city_name']}) | "
        f"mean distance = {float(best['mean_distance_km']):.3f} km"
    )

mapping_df = pd.DataFrame(results).sort_values('city').reset_index(drop=True)

# Speichern
os.makedirs(CLEANED_BASE, exist_ok=True)

out_parquet = f'{CLEANED_BASE}/city_to_weather_station_mapping.parquet'
out_csv     = f'{CLEANED_BASE}/city_to_weather_station_mapping.csv'

mapping_df.to_parquet(out_parquet, index=False)
mapping_df.to_csv(out_csv, index=False)

print('\n╔════════════════════════════════════════════════════════════╗')
print('║   MAPPING ABGESCHLOSSEN                                   ║')
print('╚════════════════════════════════════════════════════════════╝')

display(mapping_df)

print(f'\nGespeichert:')
print(f'  {out_parquet}')
print(f'  {out_csv}')

╔════════════════════════════════════════════════════════════╗
║   MAPPING ALLER STÄDTE → REPRÄSENTATIVSTE WETTERSTATION   ║
╚════════════════════════════════════════════════════════════╝

Gefundene Städte in /content/data/demand/: 135
['AW-bike (VRM)', 'Achern', 'Adenau', 'Altenahr', 'Amprion Brauweiler', 'Amprion Rommerskirchen', 'Anröchte', 'Appenweier', 'Bad Breisig', 'Bad Neuenahr-Ahrweiler'] ...

Geo geladen aus cleaned: /content/drive/MyDrive/BA_Colab/cleaned/geo_information/geo_information.parquet
Wetterstationen mit Geo-Koordinaten: 120
⚠️ Übersprungen (kein cleaned demand): AW-bike (VRM)
⚠️ Übersprungen (kein cleaned demand): Achern
⚠️ Übersprungen (kein cleaned demand): Adenau
⚠️ Übersprungen (kein cleaned demand): Altenahr
⚠️ Übersprungen (kein cleaned demand): Amprion Brauweiler
⚠️ Übersprungen (kein cleaned demand): Amprion Rommerskirchen
⚠️ Übersprungen (kein cleaned demand): Anröchte
⚠️ Übersprungen (kein cleaned demand): Appenweier
⚠️ Übersprungen (kein cleaned deman

,city,nearest_weather_location_id,weather_station_city_name,weather_city_station_FD,w_lat,w_long,mean_distance_km
0,Bochum,292353,Bochum,Nordrhein-Westfalen,51.503,7.229,5.209
1,Braunschweig,292316,Brunswick,Niedersachsen,52.291,10.446,6.513
2,Dortmund,292388,Waltrop,Nordrhein-Westfalen,51.597,7.405,108.156
3,Duisburg,292321,Duisburg,Nordrhein-Westfalen,51.509,6.702,10.051
4,Essen,292377,Essen,Nordrhein-Westfalen,51.404,6.968,6.186
5,Freiburg,292302,Freiburg im Breisgau,Baden-Württemberg,48.023,7.834,5.095
6,Gießen,292325,Gießen,Hessen,50.602,8.644,3.880
7,Heidelberg,292357,Eppelheim,Baden-Württemberg,49.377,8.618,5.772
8,Kaiserslautern,5674,Kaiserslautern,Rheinland-Pfalz,49.426,7.756,2.374
9,Kassel,292411,Kassel,Hessen,51.283,9.359,9.631



Gespeichert:
  /content/drive/MyDrive/BA_Colab/cleaned/city_to_weather_station_mapping.parquet
  /content/drive/MyDrive/BA_Colab/cleaned/city_to_weather_station_mapping.csv
